# 02 - Univariate GARCH models

This notebook fits the univariate GARCH-family models required before the DCC step.

It uses the existing project package:

- data/preprocessing from `garch_btc_sp.data`,
- statistical/model code from `garch_btc_sp.models`,
- assets already used in the repository: `BTC`, `SP500`, `VIX`, `OIL`, `GOLD`.

No generated CSV/PNG files are committed by this notebook.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Project root: {ROOT}")

Project root: C:\Users\User HP\Desktop\projekt_garch\.push_repo


In [2]:
import pandas as pd
from IPython.display import display

from garch_btc_sp.data.preprocessing import build_yahoo_returns
from garch_btc_sp.models import GARCH_VARIANTS, compare_models, fit_model_grid, select_best_by_bic

print("Available model variants:", GARCH_VARIANTS)

Available model variants: ('GARCH', 'EGARCH', 'GJR_GARCH', 'APARCH')


## 1. Load returns from the existing data pipeline

The notebook first tries to read `data/processed/returns_yahoo.parquet`, which is already produced by the repository data step. If it is missing, it falls back to `build_yahoo_returns()` from `garch_btc_sp.data.preprocessing`.

In [3]:
returns_path = ROOT / "data" / "processed" / "returns_yahoo.parquet"

if returns_path.exists():
    returns = pd.read_parquet(returns_path)
else:
    returns = build_yahoo_returns()

returns = returns.dropna()
print(returns.shape)
display(returns.head())

(2932, 5)


Ticker,BTC,OIL,GOLD,SP500,VIX
Date,,,,,
2014-09-18,-0.074643,-0.014401,-0.007073,0.004879,-0.050254
2014-09-19,-0.072402,-0.007117,-0.008521,-0.000477,0.006628
2014-09-22,0.018461,-0.009678,0.001234,-0.008046,0.122634
2014-09-23,0.080333,0.000437,0.003446,-0.005793,0.086707
2014-09-24,-0.029306,0.013452,-0.001968,0.007802,-0.117867


## 2. Fit 12 model combinations per asset

For each asset, this fits:

- GARCH(1,1),
- EGARCH(1,1),
- GJR-GARCH/TGARCH(1,1),
- APARCH(1,1),

each with `normal`, `t`, and `skewt` innovations.

In [4]:
all_comparisons = []
best_residuals = {}
best_volatility = {}

for asset in returns.columns:
    print(f"Fitting models for {asset}...")
    fitted = fit_model_grid(returns[asset])
    comparison = compare_models(fitted)
    all_comparisons.append(comparison)

    best_row = select_best_by_bic(comparison).iloc[0]
    best_model = next(
        model
        for model in fitted
        if model.variant == best_row["model"] and model.distribution == best_row["distribution"]
    )
    best_residuals[asset] = best_model.standardized_residuals.rename(asset)
    best_volatility[asset] = best_model.conditional_volatility.rename(asset)

comparison_all = pd.concat(all_comparisons, ignore_index=True)
best_models = select_best_by_bic(comparison_all)
display(best_models)

Fitting models for BTC...


Fitting models for OIL...


Fitting models for GOLD...


Fitting models for SP500...


Fitting models for VIX...


,asset,model,distribution,aic,bic,log_likelihood,resid_lb_pvalue,squared_resid_lb_pvalue
0,BTC,EGARCH,t,15569.700790,15605.601431,-7778.850395,0.000908,0.275629
1,GOLD,EGARCH,t,7773.835434,7809.736075,-3880.917717,0.643670,0.517026
2,OIL,GARCH,skewt,13133.391576,13169.292216,-6560.695788,0.425101,0.741611
3,SP500,APARCH,skewt,7184.429719,7232.297240,-3584.214860,0.251460,0.229345
4,VIX,APARCH,skewt,19481.298147,19529.165667,-9732.649073,0.313967,0.243139


## 3. Model comparison table

Lower BIC is used for model selection. Ljung-Box p-values are reported for standardized residual diagnostics.

In [5]:
display(comparison_all.sort_values(["asset", "bic"]))

,asset,model,distribution,aic,bic,log_likelihood,resid_lb_pvalue,squared_resid_lb_pvalue
0,BTC,EGARCH,t,15569.700790,15605.601431,-7778.850395,0.000908,0.275629
1,BTC,APARCH,t,15567.324326,15609.208407,-7776.662163,0.001508,0.452165
2,BTC,EGARCH,skewt,15571.329364,15613.213445,-7778.664682,0.000905,0.271263
3,BTC,APARCH,skewt,15568.709258,15616.576779,-7776.354629,0.001486,0.442386
4,BTC,GARCH,t,15601.869657,15631.786858,-7795.934829,0.001807,0.391934
5,BTC,GJR_GARCH,t,15603.051312,15638.951953,-7795.525656,0.002269,0.340740
6,BTC,GARCH,skewt,15603.736176,15639.636816,-7795.868088,0.001792,0.389100
7,BTC,GJR_GARCH,skewt,15604.919344,15646.803424,-7795.459672,0.002239,0.337646
8,BTC,EGARCH,normal,16281.068728,16310.985928,-8135.534364,0.001489,0.654455
9,BTC,GARCH,normal,16293.586305,16317.520065,-8142.793153,0.002789,0.690167


## 4. Standardized residuals for DCC

These residuals are the output needed by the DCC-GARCH stage.

In [6]:
standardized_residuals = pd.concat(best_residuals.values(), axis=1).dropna()
conditional_volatility = pd.concat(best_volatility.values(), axis=1).dropna()

print("Standardized residuals:", standardized_residuals.shape)
display(standardized_residuals.head())

print("Conditional volatility:", conditional_volatility.shape)
display(conditional_volatility.head())

Standardized residuals: (2932, 5)


,BTC,OIL,GOLD,SP500,VIX
Date,,,,,
2014-09-18,-1.559626,-0.856643,-0.915193,0.465968,-0.548890
2014-09-19,-1.352602,-0.425202,-1.097253,-0.082598,0.016348
2014-09-22,0.286302,-0.589502,0.084833,-0.987235,1.483416
2014-09-23,1.394731,0.018096,0.362891,-0.618278,0.757563
2014-09-24,-0.485622,0.836473,-0.313987,0.718203,-1.007587


Conditional volatility: (2932, 5)


,BTC,OIL,GOLD,SP500,VIX
Date,,,,,
2014-09-18,4.879963,1.697406,0.831274,0.990375,10.099717
2014-09-19,5.461130,1.706571,0.825321,0.898098,8.841563
2014-09-22,5.936102,1.665382,0.823635,0.841748,7.917634
2014-09-23,5.654623,1.642740,0.802143,0.979766,10.761396
2014-09-24,6.336589,1.591504,0.796959,1.049502,12.212281
